# Cervical Cancer Risk Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `risk`

## 2 · Project Overview

We predict **cervical cancer risk** (high vs. low) based on gynecological health indicators including age, reproductive history, smoking, contraceptive use, STD history, and sexual behavior.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a patient's age, reproductive history, smoking exposure, contraceptive use, STD count, prior diagnoses, and sexual history, predict whether they are at high risk for cervical cancer.

## 5 · Why This Project Matters

- **Early cancer detection** saves lives through timely screening.
- Risk prediction enables **targeted screening programs**.
- Understanding risk factors supports **public health education**.
- Demonstrates healthcare binary classification with class imbalance.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,000 |
| **Features** | age, num_pregnancies, smoking_years, hormonal_contraceptive_years, iud_years, num_std, num_diagnoses, first_sexual_intercourse_age, num_partners |
| **Target** | risk (0 = low risk, 1 = high risk) |
| **Class balance** | ~80/20 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "risk"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: risk
Save dir: E:\Github\Machine-Learning-Projects\Classification\Cervical Cancer Risk Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 6,000 patients with gynecological health indicators and cancer risk.

In [4]:
np.random.seed(SEED)
n = 6000
age = np.random.randint(18, 65, n)
num_pregnancies = np.random.poisson(2, n).clip(0, 10)
smoking_years = np.where(np.random.binomial(1, 0.25, n), np.random.randint(1, 30, n), 0)
hormonal_contraceptive_years = np.round(np.random.exponential(3, n).clip(0, 20), 1)
iud_years = np.round(np.random.exponential(1.5, n).clip(0, 15), 1)
num_std = np.random.poisson(0.5, n).clip(0, 5)
num_diagnoses = np.random.poisson(0.3, n).clip(0, 4)
first_sexual_intercourse_age = np.random.randint(14, 25, n)
num_partners = np.random.poisson(3, n).clip(1, 15)

score = (0.02 * age + 0.1 * num_pregnancies + 0.03 * smoking_years
         + 0.05 * hormonal_contraceptive_years + 0.08 * iud_years
         + 0.3 * num_std + 0.4 * num_diagnoses
         - 0.05 * first_sexual_intercourse_age + 0.08 * num_partners
         + np.random.normal(0, 1.0, n))
risk = (score > np.percentile(score, 80)).astype(int)

df = pd.DataFrame({"age": age, "num_pregnancies": num_pregnancies,
                    "smoking_years": smoking_years,
                    "hormonal_contraceptive_years": hormonal_contraceptive_years,
                    "iud_years": iud_years, "num_std": num_std,
                    "num_diagnoses": num_diagnoses,
                    "first_sexual_intercourse_age": first_sexual_intercourse_age,
                    "num_partners": num_partners, "risk": risk})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['risk'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (6000, 10)
Class balance:
risk
0    0.8
1    0.2
Name: proportion, dtype: float64


,age,num_pregnancies,smoking_years,hormonal_contraceptive_years,iud_years,num_std,num_diagnoses,first_sexual_intercourse_age,num_partners,risk
0,56,2,0,1.3,0.6,0,0,14,1,0
1,46,6,0,2.5,1.5,0,0,24,3,0
2,32,0,11,2.3,2.5,0,0,15,7,0
3,60,1,0,1.5,4.9,2,1,21,4,1
4,25,3,0,3.1,2.1,0,0,18,4,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (6000, 10)

Missing values:
age                             0
num_pregnancies                 0
smoking_years                   0
hormonal_contraceptive_years    0
iud_years                       0
num_std                         0
num_diagnoses                   0
first_sexual_intercourse_age    0
num_partners                    0
risk                            0
dtype: int64

Duplicate rows: 0

Dtypes:
age                               int32
num_pregnancies                   int32
smoking_years                     int32
hormonal_contraceptive_years    float64
iud_years                       float64
num_std                           int32
num_diagnoses                     int32
first_sexual_intercourse_age      int32
num_partners                      int32
risk                              int64
dtype: object

Target 'risk' confirmed. Value counts:
risk
0    4800
1    1200
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age", "smoking_years", "hormonal_contraceptive_years", "num_std", "num_diagnoses", "num_partners"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Cancer Risk", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `risk`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Cervical Cancer Risk Distribution")
ax.set_xticklabels(["Low Risk (0)", "High Risk (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"High risk rate: {df[TARGET].mean():.1%}")

High risk rate: 20.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (4800, 9), Test: (1200, 9)
Train class distribution:
risk
0    0.8
1    0.2
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: None (all numeric features).
- **Scaling**: Not needed for tree models.
- **Class balance**: ~20% high risk (imbalanced).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["risk_exposure_years"] = X_train["smoking_years"] + X_train["hormonal_contraceptive_years"]
X_test["risk_exposure_years"] = X_test["smoking_years"] + X_test["hormonal_contraceptive_years"]

X_train["sexual_risk_score"] = X_train["num_std"] + X_train["num_partners"] * 0.5
X_test["sexual_risk_score"] = X_test["num_std"] + X_test["num_partners"] * 0.5

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (11): ['age', 'num_pregnancies', 'smoking_years', 'hormonal_contraceptive_years', 'iud_years', 'num_std', 'num_diagnoses', 'first_sexual_intercourse_age', 'num_partners', 'risk_exposure_years', 'sexual_risk_score']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.8033
F1       : 0.2081

              precision    recall  f1-score   support

           0       0.82      0.97      0.89       960
           1       0.53      0.13      0.21       240

    accuracy                           0.80      1200
   macro avg       0.68      0.55      0.55      1200
weighted avg       0.76      0.80      0.75      1200



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
NearestCentroid             0.697500           0.684375  0.743746  0.724457   0.786733  0.697500    0.021105
GaussianNB                  0.768333           0.609896  0.730729  0.761073   0.755268  0.768333    0.027663
XGBClassifier               0.792500           0.589063  0.663012  0.766961   0.758569  0.792500    0.178279
DecisionTreeClassifier      0.727500           0.589063  0.589063  0.731326   0.735485  0.727500    0.040101
LGBMClassifier              0.797500           0.575000  0.700977  0.763063   0.757994  0.797500    4.096083
CatBoostClassifier          0.800833           0.567708  0.723880  0.760778   0.759831  0.800833    3.278588
ExtraTreeClassifier         0.710000           0.565625  0.565625  0.715069   0.720629  0.7100

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.7817
F1: 0.3316


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.1661  (1.7s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[48]	valid_0's binary_logloss: 0.444555
LightGBM F1: 0.1958  (0.9s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.7817  0.3316     0.4276  0.2708
Logistic Regression    0.8033  0.2081     0.5345  0.1292
LightGBM               0.8083  0.1958     0.6087  0.1167
CatBoost               0.7992  0.1661     0.4898  0.1000

Top 3 models: ['FLAML', 'Logistic Regression', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.7817
    F1       : 0.3316
    Precision: 0.4276
    Recall   : 0.2708

  Logistic Regression:
    Accuracy : 0.8033
    F1       : 0.2081
    Precision: 0.5345
    Recall   : 0.1292

  LightGBM:
    Accuracy : 0.8083
    F1       : 0.1958
    Precision: 0.6087
    Recall   : 0.1167


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       960
           1       0.43      0.27      0.33       240

    accuracy                           0.78      1200
   macro avg       0.63      0.59      0.60      1200
weighted avg       0.75      0.78      0.76      1200


Total errors: 262 / 1200 (21.8%)

Sample misclassifications:
      age  num_pregnancies  smoking_years  hormonal_contraceptive_years  iud_years  num_std  num_diagnoses  first_sexual_intercourse_age  num_partners  risk_exposure_years  sexual_risk_score  actual  predicted  correct
1218   36                3              0                           2.8        0.1        0              0                            17             4                  2.8                2.0       1          0    False
4947   42                4             24                           0.7        4.5        0              2                         

## 25 · Interpretation and Business Insight

**Key findings:**
- **Prior diagnoses** and **STD count** are the strongest risk indicators.
- **Smoking exposure** adds significant cumulative risk.
- **Number of sexual partners** correlates with STD exposure.
- **Age** provides moderate signal (cumulative exposure).

**Business takeaway:** Prioritize screening for patients with prior diagnoses, multiple STDs, and long smoking histories.

## 26 · Limitations

1. Synthetic data — real cervical cancer has complex etiology.
2. No HPV status (the primary actual cause).
3. No Pap smear or colposcopy results.
4. Missing socioeconomic factors.
5. Binary risk is an oversimplification.

## 27 · How to Improve This Project

1. Include HPV test results (primary risk factor).
2. Add Pap smear history and colposcopy data.
3. Use real UCI Cervical Cancer dataset.
4. Model risk as continuous probability.
5. Add genetic and family history features.

## 28 · Production Considerations

- Deploy as a screening prioritization tool.
- Output risk probability, not just binary label.
- Must include clinical validation before deployment.
- Integrate with electronic health records.
- Requires regulatory approval (medical device).

## 29 · Common Mistakes

1. Deploying without clinical validation.
2. Using accuracy on 80/20 imbalanced data.
3. Missing HPV — the actual primary cause.
4. Not calibrating probabilities for medical decisions.
5. Treating the model as diagnostic (it's a screening tool).

## 30 · Mini Challenge / Exercises

1. Try SMOTE to balance the 80/20 split.
2. Plot ROC and PR curves — which is more informative here?
3. Set a high-recall threshold (catch 95% of at-risk).
4. Remove `num_diagnoses` (potential leakage?) and compare.
5. Analyze false negatives — what do missed cases look like?

## 31 · Final Summary / Key Takeaways

1. **Prior diagnoses** and **STD history** are the strongest risk signals.
2. **Smoking** adds cumulative risk over years.
3. **Class imbalance** (80/20) requires PR-AUC and recall focus.
4. **HPV status** (missing here) is the actual primary cause.
5. **Medical ML** requires clinical validation before deployment.
6. **High recall** is critical — missing cancer is worse than false alarms.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Cervical Cancer Risk Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7992,
    "f1": 0.1661,
    "precision": 0.4898,
    "recall": 0.1
  },
  "LightGBM": {
    "accuracy": 0.8083,
    "f1": 0.1958,
    "precision": 0.6087,
    "recall": 0.1167
  },
  "Logistic Regression": {
    "accuracy": 0.8033,
    "f1": 0.2081,
    "precision": 0.5345,
    "recall": 0.1292
  },
  "FLAML": {
    "accuracy": 0.7817,
    "f1": 0.3316,
    "precision": 0.4276,
    "recall": 0.2708
  }
}
